In [ ]:


import pandas as pd
import numpy as np



def load_sensor_data(filepath: str) -> pd.DataFrame:
    
    df = pd.read_csv(r"C:\Users\minhn\Downloads\complex.csv")
    df.columns = df.columns.str.strip()          # remove accidental whitespace
    print(f"[LOAD] Loaded {len(df)} rows, columns: {list(df.columns)}")
    return df

In [21]:

def handle_missing_values(df: pd.DataFrame, method: str = "interpolate") -> pd.DataFrame:
    """
    Fill missing sensor readings.
    method: 'interpolate' (default) | 'ffill' | 'drop'
    """
    missing_before = df.isnull().sum().sum()

    if method == "interpolate":
        df = df.interpolate(method="linear", limit_direction="both")
    elif method == "ffill":
        df = df.ffill().bfill()
    elif method == "drop":
        df = df.dropna()
    else:
        raise ValueError(f"Unknown method: {method}")

    missing_after = df.isnull().sum().sum()
    print(f"[MISSING] Fixed {missing_before - missing_after} missing values using '{method}'")
    return df

In [22]:

def fix_timestamps(df: pd.DataFrame, time_col: str = "time") -> pd.DataFrame:
    df[time_col] = pd.to_numeric(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col])

    dupes_before = df.duplicated(subset=[time_col]).sum()
    df = df.drop_duplicates(subset=[time_col])
    df = df.sort_values(time_col).reset_index(drop=True)

    print(f"[TIMESTAMPS] Removed {dupes_before} duplicate timestamps. Sorted {len(df)} rows ascending.")
    return df

In [23]:
def remove_outliers(df: pd.DataFrame,
                    sensor_cols: list,
                    iqr_factor: float = 3.0) -> pd.DataFrame:

    total_replaced = 0
    for col in sensor_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - iqr_factor * IQR
        upper = Q3 + iqr_factor * IQR

        mask = (df[col] < lower) | (df[col] > upper)
        n_outliers = mask.sum()
        df.loc[mask, col] = np.nan
        total_replaced += n_outliers

    # Re-interpolate any NaNs we just introduced
    df[sensor_cols] = df[sensor_cols].interpolate(method="linear", limit_direction="both")
    print(f"[OUTLIERS] Replaced {total_replaced} outlier values (IQR factor={iqr_factor})")
    return df

In [24]:


def align_to_common_index(df: pd.DataFrame, time_col: str = "time", freq: int = 1) -> pd.DataFrame:
    df = df.set_index(time_col)

    start = int(df.index.min())
    end = int(df.index.max())

    common_index = range(start, end + 1, freq)
    df = df.reindex(common_index)
    df = df.interpolate(method="linear")

    df = df.reset_index().rename(columns={"index": time_col})

    print(f"[ALIGN] Resampled to {len(df)} rows using freq={freq}.")
    return df

In [25]:
def validate_output(df: pd.DataFrame, time_col: str = "time") -> pd.DataFrame:
    """Assert the dataframe is sorted, has no nulls, and dtypes are float."""
    assert df[time_col].is_monotonic_increasing, "Time column is NOT sorted!"
    assert df.isnull().sum().sum() == 0, "DataFrame still contains NaN values!"

    sensor_cols = [c for c in df.columns if c != time_col]
    for col in sensor_cols:
        df[col] = df[col].astype(np.float64)

    print(f"[VALIDATE] ✓ Sorted | ✓ No nulls | ✓ Consistent float64 dtypes")
    print(f"[VALIDATE] Output shape: {df.shape}")
    return df

In [26]:


def run_pipeline(input_path: str, output_path: str = "clean_sensor_data.csv"):
    print("=" * 55)
    print("  SENSOR PREPROCESSING PIPELINE")
    print("=" * 55)

    # Step 1 – Load
    df = load_sensor_data(input_path)

    # Identify sensor columns (everything except time)
    time_col = "time"
    sensor_cols = [c for c in df.columns if c != time_col]
    print(f"[INFO] Sensor channels: {sensor_cols}")

    # Step 2 – Fix timestamps
    df = fix_timestamps(df, time_col=time_col)

    # Step 3 – Handle missing values
    df = handle_missing_values(df, method="interpolate")

    # Step 4 – Remove outliers
    df = remove_outliers(df, sensor_cols=sensor_cols, iqr_factor=3.0)

    # Step 5 – Align to common time index
    df = align_to_common_index(df, time_col=time_col, freq=1)

    # Step 6 – Validate
    df = validate_output(df, time_col=time_col)

    # Step 7 – Save
    df.to_csv(output_path, index=False)
    print(f"\n[DONE] Clean dataset saved → {output_path}")
    print("=" * 55)
    return df

In [27]:
if __name__ == "__main__":
    INPUT_FILE  = "complex.csv"      # ← your raw sensor file
    OUTPUT_FILE = "clean_sensor_data1.csv"

    clean_df = run_pipeline(INPUT_FILE, OUTPUT_FILE)
    print(clean_df.head())

  SENSOR PREPROCESSING PIPELINE
[LOAD] Loaded 1008 rows, columns: ['time', 's1', 's2', 's3']
[INFO] Sensor channels: ['s1', 's2', 's3']
[TIMESTAMPS] Removed 0 duplicate timestamps. Sorted 1008 rows ascending.
[MISSING] Fixed 0 missing values using 'interpolate'
[OUTLIERS] Replaced 0 outlier values (IQR factor=3.0)
[ALIGN] Resampled to 1008 rows using freq=1.
[VALIDATE] ✓ Sorted | ✓ No nulls | ✓ Consistent float64 dtypes
[VALIDATE] Output shape: (1008, 4)

[DONE] Clean dataset saved → clean_sensor_data1.csv
   time        s1       s2        s3
0     0  1.000000  2.00000  0.700000
1     1  1.010000  1.99995  0.707000
2     2  1.019999  1.99980  0.713999
3     3  1.029996  1.99955  0.720997
4     4  1.039989  1.99920  0.727993


In [28]:
import os
print(os.getcwd())

C:\Users\New
